# STITCH Impute: PARAMNAME id

In [ ]:
id <- "paramset"
plotdir <- "plots/"
compare <- "~/impute.compare.stats"
info <- "~/impute.infoscore"
model <- "haploid"
usebx <- 'true'
bxlimit <- 50000
k <- 15
s <- 20
ngen <- 5
extra <- "None"

In [ ]:
cat(format(Sys.time(), '🗓️ %d %B, %Y 🕔 %H:%M'))

In [ ]:
library(dplyr)
library(tidyr)
library(DT)
library(highcharter)
library(scales)

In [ ]:
comparefile <- compare
tryCatch(
  expr = {
    gcts <- read.table(comparefile, header = F)[, c(2, 23, 25, 24, 27)]
  },
  error = function(e){ 
    cat("Stats file is empty. Perhaps there were no biallelelic SNPs that bcftools identified.\n")
    knitr::knit_exit()
  }
)
names(gcts) <- c("sample","RefHom", "Het",	"AltHom", "missing")

gcts_long <- pivot_longer(gcts, -1, names_to = "Conversion", values_to = "Count")

In [ ]:
infofile <- info
#infofile <- "/home/pdimens/Documents/harpy/Impute/high_ngen/reports/data/impute.infoscore"
tryCatch(
  expr = {
    infos <- read.table(infofile, header = F, col.names = c("contig", "position", "info"))
  },
  error = function(e){ 
    cat("Info score file is empty. Perhaps there were no SNPs that bcftools identified.\n")
    knitr::knit_exit()
  }
)

## Param Information

In [ ]:
list(
  color = "light",
  value = id
)

In [ ]:
list(
  color = "#dfdfdf",
  value = model
)

In [ ]:
list(
  color = "#dfdfdf",
  value = usebx
)

In [ ]:
list(
  color = "#dfdfdf",
  value = bxlimit
)

In [ ]:
list(
  color = "#dfdfdf",
  value = k
)

In [ ]:
list(
  color = "#dfdfdf",
  value = s
)

In [ ]:
list(
  color = "#dfdfdf",
  value = ngen
)

##
Extra parameters provided: `r extra`

## General Information

In [ ]:
gcts_minmax <- filter(gcts_long, Conversion != "missing") %>% summarise(min = min(Count), max = max(Count))

In [ ]:
list(
  color = "light",
  value = prettyNum(nrow(gcts), big.mark = ",")
)

In [ ]:
list(
  color = "info",
  value = prettyNum(gcts_minmax$min, big.mark = ",")
)

In [ ]:
list(
  color = "info",
  value = prettyNum(gcts_minmax$max, big.mark = ",")
)

In [ ]:
list(
  color = "info",
  value = round(mean(infos$info), digits = 2)
)

In [ ]:
list(
  color = "info",
  value = round(sd(infos$info), digits = 2)
)

## INFO Scores
::: {.card title="Global INFO_SCORE Distribution"}
This is the global frequency distribution of `INFO_SCORE` values for
the variants in the VCF file.  This value ranges from `0` to `1.0`, with `0` being
very poor, and `1.0` being very good. It shows up to 30 of the largest
contigs present in the VCF file.

In [ ]:
h <- hist(infos$info, breaks = 20, plot = F)
h <- data.frame(info = round(h$breaks[1:length(h$breaks)-1], 2), freq = round(h$counts / sum(h$counts)  * 100, 2))
hchart(h, "areaspline", hcaes(x = info, y = freq), name = "% sites", marker = list(enabled = FALSE)) |>
  hc_yAxis(title = list(text = "% of sites")) |>
  hc_xAxis(title = list(text = "INFO_SCORE")) |>
  hc_title(text = "Distribution of imputation INFO_SCORE values") |>
  hc_tooltip(crosshairs = TRUE, animation = FALSE,
    formatter = JS("function () {return 'INFO_SCORE <b>' + this.x + '</b><br><b>' + this.y + '% of sites</b>';}") 
  ) |>
  hc_exporting(enabled = T, filename = "impute.infoscores",
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
)

:::

###
::: {.card title="Per-Contig INFO_SCORE Distribution"}
This is the per-contig frequency distribution of `INFO_SCORE` values
for the variants in the VCF file.  This value ranges from `0` to `1.0`, with `0` being
very poor, and `1.0` being very good. It shows up to 30 of the largest
contigs present in the VCF file.

In [ ]:
dat <- data_to_boxplot(infos, info, contig, name = "INFO_SCORE", animation = FALSE)

highchart() |>
  hc_xAxis(type = "category") |>
  hc_colors("#8487bb") |>
  hc_add_series_list(dat)

:::

# Filtering
## Filtering Expectations
<h2> Filtering Expectations </h2>
`STITCH` outputs a novel `INFO/INFO_SCORE` tag in the resulting VCF files that represents the
"quality" of the imputation. This value ranges from `0` to `1.0`, with `0` being
very poor, and `1.0` being very good. You will want to filter this file to remove
genotypes with a low `INFO_SCORE`, which can be done with `bcftools`.
Here is an example snippet of keeping loci with an `INFO_SCORE` of at least `0.2`: 

```bash
# use -Ob for bcf output (smaller file size)

bcftools view -Ob -i 'INFO/INFO_SCORE >= 0.2' file.vcf > filtered.file.bcf
```

##
::: {.card title="SNPs Remaining After Filtering" expandable="true"}
This is the number of SNP sites that may remain
if filtering at different minimum `INFO/INFO_SCORE` thresholds.

In [ ]:
filt_remaining <- function(x, thresh){
  sum(x >= thresh)
}
thresholds <- seq(0, 1, 0.01) 
filt_results <- data.frame(
  "threshold" = thresholds,
  "remaining" = sapply(thresholds, function(x){filt_remaining(infos$info, x)})
)

In [ ]:
hchart(filt_results, "areaspline", hcaes(x = threshold, y = remaining), color = "#8484bd", name = "SNPs remaining", marker = list(enabled = FALSE), animation = F) |>
  hc_title(text = "SNPs remaning after INFO filtering") |>
  hc_xAxis(min = 0, max = 1, title = list(text = "INFO score filtering threshold")) |>
  hc_yAxis(title = list(text = "SNPs remaning after filtering")) |>
  hc_tooltip(
    crosshairs = TRUE,
    pointFormat= 'filter threshold: <b>{point.x}</b><br>SNPs remaining: <b>{point.y}</b>'
    ) |>
  hc_exporting(enabled = T, filename = "snps.INFO.filtering",
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
  )

:::

###
::: {.card title="SNPs Remaining Per Contig After Filtering" expandable="true"}
This is the number of SNP sites that may remain for each contig
if filtering at various minimum `INFO/INFO_SCORE` thresholds.
Each column after `SNPs` refers to a filtering threshold
and the numbers within that column are the number of SNPs that would
be kept if filtering out SNPs with an `INFO/INFO_SCORE` value below 
it (_i.e._ `0.2` is the number of SNPs with a score `>= 0.2` ).

In [ ]:
filterthresh <- infos %>% group_by(contig) %>%
  summarise(
    totals = length(info),
    twenty = sum(info >= 0.2),
    twentyfive = sum(info >= 0.25),
    thirty = sum(info >= 0.30),
    forty = sum(info >= 0.40),
    fifty = sum(info >= 0.50),
    seventyfive = sum(info >= 0.75),
    ninety = sum(info >= 0.90),
    )

tots <- colSums(filterthresh[, -1])
filterthresh <- rbind(c("all", tots), filterthresh) 
column_description <- c(
  "name of the contig",
  "total number of SNPs on the contig",
  "number of SNPs remaining after filtering at INFO>=0.2",
  "number of SNPs remaining after filtering at INFO>=0.25",
  "number of SNPs remaining after filtering at INFO>=0.3",
  "number of SNPs remaining after filtering at INFO>=0.4",
  "number of SNPs remaining after filtering at INFO>=0.5",
  "number of SNPs remaining after filtering at INFO>=0.75",
  "number of SNPs remaining after filtering at INFO>=0.90"
)

headerCallback <- c(
  "function(thead, data, start, end, display){",
  paste0(" var tooltips = ['", paste(column_description, collapse = "','"), "'];"),
  "  for(var i=0; i<tooltips.length; i++){",
  "    $('th:eq('+i+')',thead).attr('title', tooltips[i]);",
  "  }",
  "}"
)

DT::datatable(
  filterthresh,
  rownames = FALSE,
  extensions = "Buttons",
  colnames = c("Contig", "SNPs", "0.20", "0.25", "0.30", "0.4", "0.5", "0.75", "0.90"),
  fillContainer = T,
  caption = 'Number of SNPs to be KEPT by filtering',
  options = list(
    dom = "Brtp",
    buttons = c("csv"),
    scrollX = TRUE,
    headerCallback = JS(headerCallback)
  )
)

:::

# Individual Statistics
## Ind stats desc
::: {.card title="Imputations Per Sample"}
This chart visualizes the count and proportion of `missing` genotypes per sample imputed
into other genotypes. Following standard VCF convention:

Ref
: The reference allele (from the reference genome)

Alt
: The alternative allele

Missing
: How many SNPs remain missing (failed to impute) for that individual

As an example, the `Homozygous Ref` is the number of missing genotypes that were imputed into
a homozygous genotype for the reference allele. The table on the other tab is the
underlying data used to create the visualization.
:::

##

In [ ]:
gcts_long$Conversion <- gsub("AltHom","Homozygous Alt", gcts_long$Conversion)
gcts_long$Conversion <- gsub("Het","Heterozygote", gcts_long$Conversion)
gcts_long$Conversion <- gsub("RefHom","Homozygous Ref", gcts_long$Conversion)
gcts_long$Conversion <- gsub("missing","Missing", gcts_long$Conversion)
gcts_long$Conversion <- factor(gcts_long$Conversion, levels = c("Heterozygote", "Homozygous Ref", "Homozygous Alt", "Missing"), ordered = T)
gcts_long$conv_num <- as.integer(gcts_long$Conversion) -1

In [ ]:
highchart() |>
  hc_add_series(
    data = gcts_long,
    type = "scatter",
    hcaes(x = Count, y = conv_num, color = Conversion, group = sample),
    marker = list(radius = 5, symbol = "circle"),
    animation = F
  ) |>
  hc_colors(c("#7cb3d4", "#b18ad1", "#ffd75f", "#6f6c70")) |>
  hc_title(text = "Genotypes") |>
  hc_caption(text = "Values indicate how many genotypes were imputed into which kind of genotype") |>
  hc_plotOptions(scatter = list(jitter = list(x = 0.1, y = 0.4))) |>
  hc_yAxis(
    title = list(text = "Genotypes"),
    categories = levels(gcts_long$Conversion),
    labels = list(align = "center"),
    minorGridLineWidth = 1
    ) |>
  hc_xAxis(title = list(text = "Count")) |>
  hc_tooltip(
    animation = F,
    pointFormat = '{point.Conversion}<br>Count: {point.x}'
  ) |>
  hc_exporting(enabled = T, filename = "impute.samples",
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
  ) |>
  hc_legend(enabled = FALSE)

###

In [ ]:
DT::datatable(
  gcts,
  rownames = FALSE,
  extensions = "Buttons",
  colnames = c("Sample", "Homozygous Ref", "Homozygous Alt", "Heterozygote", "Missing"),
  autoHideNavigation = T,
  fillContainer = T,
  options = list(
    dom = "Brtp",
    buttons = c("csv"),
    scrollX = TRUE
  )
)